# Breast Cancer Detection — Notebook 6
## Classical Machine Learning on the Wisconsin Diagnostic Dataset (WDBC)

Deep learning is not always the right tool: on tabular cytology features (30 features computed from fine-needle-aspirate images), classical models are strong, fast and interpretable. This notebook benchmarks **Logistic Regression, SVM (RBF), Random Forest, Gradient Boosting and XGBoost** with 5-fold cross-validation, compares them on a held-out test set, inspects feature importance, and tunes the best model.

### 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

### 2. Dataset — WDBC (built into scikit-learn)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)

# NOTE: in scikit-learn, target 0 = malignant and 1 = benign.
# Flip it so that malignant = 1, consistent with the image notebooks.
y = pd.Series(1 - data.target, name="label")

print(X.shape)
print(y.value_counts().rename({1: "malignant", 0: "benign"}))
X.head()

In [ ]:
X.describe().T.head(12)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# class balance
counts = y.value_counts().sort_index()
axes[0].bar(["benign (0)", "malignant (1)"], counts.values,
            color=["#0072B2", "#D55E00"], width=0.55)
axes[0].set_ylabel("Samples")
axes[0].set_title("Class balance")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center")

# correlation of the 10 "mean" features (single-hue sequential colormap)
mean_cols = [c for c in X.columns if "mean" in c]
sns.heatmap(X[mean_cols].corr(), cmap="Blues", annot=True, fmt=".2f",
            annot_kws={"size": 7}, ax=axes[1], cbar=False)
axes[1].set_title("Correlation — mean features")
axes[1].tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(base_path + "/plots/wdbc_eda.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train only (no leakage)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "| Test:", X_test_s.shape)

### 3. Models + 5-fold cross-validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05,
                             eval_metric="logloss", random_state=42),
}

for name, clf in models.items():
    cv_auc = cross_val_score(clf, X_train_s, y_train, cv=5, scoring="roc_auc")
    print(f"{name:20s} | CV ROC-AUC: {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

### 4. Test-set comparison

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

rows = []
fitted = {}
probs_by_model = {}

for name, clf in models.items():
    clf.fit(X_train_s, y_train)
    fitted[name] = clf

    probs = clf.predict_proba(X_test_s)[:, 1]
    preds = (probs > 0.5).astype(int)
    probs_by_model[name] = probs

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs),
    })

comparison = (pd.DataFrame(rows)
                .sort_values("roc_auc", ascending=False)
                .reset_index(drop=True))
comparison.round(4)

In [ ]:
from sklearn.metrics import roc_curve

# fixed colorblind-safe palette, one color per model (identity, fixed order)
palette = ["#0072B2", "#D55E00", "#009E73", "#56B4E9", "#CC79A7"]

plt.figure(figsize=(7, 6))
for color, (name, probs) in zip(palette, probs_by_model.items()):
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_m = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {auc_m:.3f})")

plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — WDBC test set")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(base_path + "/plots/wdbc_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

best_name = comparison.iloc[0]["model"]
best_model = fitted[best_name]

probs = probs_by_model[best_name]
preds = (probs > 0.5).astype(int)

print("Best model:", best_name)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n")
print(classification_report(y_test, preds, target_names=["benign", "malignant"]))

### 5. Feature importance (interpretability for the paper)

In [ ]:
rf = fitted["Random Forest"]

importance = (pd.Series(rf.feature_importances_, index=X.columns)
                .sort_values(ascending=True)
                .tail(15))

plt.figure(figsize=(8, 6))
plt.barh(importance.index, importance.values, color="#0072B2", height=0.6)
plt.xlabel("Importance (Gini)")
plt.title("Random Forest — top 15 features")
plt.tight_layout()
plt.savefig(base_path + "/plots/wdbc_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Hyperparameter tuning of the strongest family

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05, 0.1],
}

grid = GridSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid, cv=5, scoring="roc_auc", n_jobs=-1)

grid.fit(X_train_s, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

tuned_probs = grid.best_estimator_.predict_proba(X_test_s)[:, 1]
tuned_preds = (tuned_probs > 0.5).astype(int)
print("Tuned test ROC-AUC:", round(roc_auc_score(y_test, tuned_probs), 4))
print("Tuned test accuracy:", round(accuracy_score(y_test, tuned_preds), 4))

### 7. Save artifacts to Drive

In [ ]:
comparison.to_csv(base_path + "/results/wdbc_model_comparison.csv", index=False)
print("Comparison table saved")
comparison.round(4)

In [ ]:
import joblib

joblib.dump(best_model, base_path + "/models/wdbc_best_model.joblib")
joblib.dump(scaler, base_path + "/models/wdbc_scaler.joblib")
joblib.dump(grid.best_estimator_, base_path + "/models/wdbc_xgb_tuned.joblib")

print("Models + scaler saved")

In [ ]:
results_text = f"""DATASET: Wisconsin Diagnostic Breast Cancer (WDBC)
TASK: binary classification (malignant = 1, benign = 0)
SPLIT: 80/20 stratified, StandardScaler fit on train only
CV: 5-fold ROC-AUC on the training split

TEST-SET COMPARISON:
{comparison.round(4).to_string(index=False)}

BEST MODEL: {best_name}

TUNED XGBOOST:
params: {grid.best_params_}
CV ROC-AUC: {grid.best_score_:.4f}
Test ROC-AUC: {roc_auc_score(y_test, tuned_probs):.4f}
"""

with open(base_path + "/results/wdbc_results.txt", "w") as f:
    f.write(results_text)

print(results_text)

In [ ]:
import json

config = {
    "dataset": "WDBC (sklearn load_breast_cancer)",
    "n_samples": int(len(X)),
    "n_features": int(X.shape[1]),
    "label_convention": "malignant = 1 (sklearn target flipped)",
    "split": "80/20 stratified",
    "scaling": "StandardScaler (fit on train)",
    "cv": "5-fold, roc_auc",
    "models": list(models.keys()),
    "threshold": 0.5,
    "seed": 42
}

with open(base_path + "/results/wdbc_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")